### Competition EDA

In [1]:
import re
import pandas as pd
import statistics

In [2]:
data = pd.read_csv("../data/raw/train.csv")

In [3]:
data

,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret
...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates


In [4]:
data["prompt_eda"] = data.prompt.str.split('.').apply(lambda x: x[0])

In [5]:
print(data.prompt_eda.unique())

<ArrowStringArray>
['In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers',
                       'In Alice's Wonderland, secret encryption rules are used on text',
 'In Alice's Wonderland, numbers are secretly converted into a different numeral system',
            'In Alice's Wonderland, a secret unit conversion is applied to measurements',
           'In Alice's Wonderland, the gravitational constant has been secretly changed',
   'In Alice's Wonderland, a secret set of transformation rules is applied to equations']
Length: 6, dtype: str


In [6]:
task_classes = {
    "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers":  "bit manipulation",
    "In Alice's Wonderland, secret encryption rules are used on text": "encryption",
    "In Alice's Wonderland, numbers are secretly converted into a different numeral system": "conversion to diff numeral system",
    "In Alice's Wonderland, a secret unit conversion is applied to measurements": "unit conversion",
    "In Alice's Wonderland, the gravitational constant has been secretly changed": "gravitational",
    "In Alice's Wonderland, a secret set of transformation rules is applied to equations": "equations transformation"
}

In [7]:
data["label"] = data.prompt_eda.map(task_classes)
data["label"].value_counts()

label
bit manipulation                     1602
gravitational                        1597
unit conversion                      1594
encryption                           1576
conversion to diff numeral system    1576
equations transformation             1555
Name: count, dtype: int64

In [8]:
### Just check data

In [9]:
data.iloc[2].prompt

"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley\npqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle\ngbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door\nbxo sfjpov pqrsfv dfjjfig -> the golden dragon follows\nnqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious secret\nNow, decrypt the following text: trb wzrswvog hffk"

In [10]:
data[data.label == "equations transformation"].sample(1).prompt.to_list()

["In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n82*17 = 7891\n83+59 = 331\n52*95 = 4741\nNow, determine the result for: 75*31"]

In [11]:
data[data.label == "bit manipulation"].sample(1).prompt.to_list()

["In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n11111110 -> 11111101\n11001101 -> 11011011\n10100101 -> 01001011\n11110101 -> 11101011\n10111111 -> 11111111\n01001000 -> 10010000\n10000011 -> 11000111\n00001110 -> 10011100\n\nNow, determine the output for: 01110100"]

In [12]:
data["label_format"] = data.prompt.str.split("\n").apply(lambda x: x[-1])

In [13]:
data.groupby("label")["label_format"].value_counts().to_dict()

{('bit manipulation', 'Now, determine the output for: 01010101'): 13,
 ('bit manipulation', 'Now, determine the output for: 11110101'): 13,
 ('bit manipulation', 'Now, determine the output for: 11001101'): 12,
 ('bit manipulation', 'Now, determine the output for: 10000001'): 12,
 ('bit manipulation', 'Now, determine the output for: 11001000'): 12,
 ('bit manipulation', 'Now, determine the output for: 10101001'): 12,
 ('bit manipulation', 'Now, determine the output for: 10001001'): 12,
 ('bit manipulation', 'Now, determine the output for: 01111110'): 11,
 ('bit manipulation', 'Now, determine the output for: 11101101'): 11,
 ('bit manipulation', 'Now, determine the output for: 11111010'): 11,
 ('bit manipulation', 'Now, determine the output for: 11100110'): 11,
 ('bit manipulation', 'Now, determine the output for: 11000110'): 11,
 ('bit manipulation', 'Now, determine the output for: 00100110'): 11,
 ('bit manipulation', 'Now, determine the output for: 00110000'): 10,
 ('bit manipulation'

In [14]:
def extract_template(text):
    if not isinstance(text, str):
        return str(text)
        
    text = text.strip()
    
    # "Now, determine the output for: <TARGET>"
    if ':' in text:
        return re.sub(r':\s*.*$', ': <TARGET>', text)
        
    # "If <NUM> @ <NUM> = <NUM>, what is X?"
    text = re.sub(r'\b\d+\b', '<NUM>', text)
    
    return text

data["pattern"] = data["label_format"].apply(extract_template)

patterns_summary = data.groupby("label")["pattern"].value_counts().to_frame("count").reset_index()

for label in patterns_summary['label'].unique():
    print(f"\n=== {label} ===")
    subset = patterns_summary[patterns_summary['label'] == label]
    for _, row in subset.iterrows():
        print(f"{row['count']:>4} | {row['pattern']}")


=== bit manipulation ===
1602 | Now, determine the output for: <TARGET>

=== conversion to diff numeral system ===
1576 | Now, write the number <NUM> in the Wonderland numeral system.

=== encryption ===
1576 | Now, decrypt the following text: <TARGET>

=== equations transformation ===
1555 | Now, determine the result for: <TARGET>

=== gravitational ===
  25 | Now, determine the falling distance for t = <NUM>.32s given d = <NUM>.<NUM>*g*t^<NUM>.
  24 | Now, determine the falling distance for t = <NUM>.82s given d = <NUM>.<NUM>*g*t^<NUM>.
  24 | Now, determine the falling distance for t = <NUM>.72s given d = <NUM>.<NUM>*g*t^<NUM>.
  23 | Now, determine the falling distance for t = <NUM>.79s given d = <NUM>.<NUM>*g*t^<NUM>.
  23 | Now, determine the falling distance for t = <NUM>.87s given d = <NUM>.<NUM>*g*t^<NUM>.
  22 | Now, determine the falling distance for t = <NUM>.45s given d = <NUM>.<NUM>*g*t^<NUM>.
  22 | Now, determine the falling distance for t = <NUM>.57s given d = <NUM>.<

### conversion to diff numeral system

In [15]:
import re

class NumeralSystemSolver:
    """conversion to diff numeral system"""
    
    def __init__(self):
        self.roman_vals = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
        self.roman_syms = ["M", "CM", "D", "CD", "C", "XC", "L", "XL", "X", "IX", "V", "IV", "I"]

    def generate_cot(self, prompt: str) -> str:
        """Chain-of-Thought"""
        target_match = re.search(r"write the number (\d+)", prompt, re.IGNORECASE)
        if not target_match:
            return "Parse Error: Target not found."
        
        target_num = int(target_match.group(1))
        examples = re.findall(r"(\d+)\s*->\s*([A-Z]+)", prompt)
        
        cot = ["Let's identify the secret numeral system used in Wonderland.\n"]
        cot.append("Looking at the examples provided:")
        
        for arab, rom in examples[:3]:
            cot.append(f"  {arab} -> {rom}")
            
        cot.append("\nThe output symbols (I, V, X, L, C, D, M) and their combinations clearly indicate standard Roman Numerals.")
        cot.append(f"\nWe need to convert the number {target_num} into Roman numerals using greedy decomposition:")
        
        remaining = target_num
        parts = []
        
        for v, s in zip(self.roman_vals, self.roman_syms):
            while remaining >= v:
                parts.append(s)
                remaining -= v
                cot.append(f"  - Subtract {v} ({s}): remainder is {remaining}.")
                
        final_roman = "".join(parts)
        cot.append(f"\nCombining the symbols gives us: {final_roman}.")
        cot.append(f"The final answer is {final_roman}.")
        
        return "\n".join(cot)

    # TODO: 
    # Добавить \\boxed в ответ?
    def extract_answer(self, cot_text: str) -> str:
        if "Parse Error" in cot_text:
            return None

        match = re.search(r"The final answer is ([A-Z]+)\.", cot_text)
        return match.group(1) if match else None

In [16]:
numeral_df = data[data['label'] == 'conversion to diff numeral system'].copy()

solver = NumeralSystemSolver()

numeral_df['generated_cot'] = numeral_df['prompt'].apply(solver.generate_cot)

numeral_df['computed_answer'] = numeral_df['generated_cot'].apply(solver.extract_answer)

numeral_df['is_correct'] = numeral_df['computed_answer'].astype(str).str.strip() == numeral_df['answer'].astype(str).str.strip()

accuracy = numeral_df['is_correct'].mean()
print(f"Accuracy by '{numeral_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'conversion to diff numeral system': 100.00%


In [17]:
numeral_df

,id,prompt,answer,prompt_eda,label,label_format,pattern,generated_cot,computed_answer,is_correct
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 38 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXXVIII,True
14,00600e6e,"In Alice's Wonderland, numbers are secretly co...",LXVII,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 67 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LXVII,True
30,00d9f682,"In Alice's Wonderland, numbers are secretly co...",C,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 100 in the Wonderland nu...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,C,True
36,0106eb4a,"In Alice's Wonderland, numbers are secretly co...",LXXXIV,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 84 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LXXXIV,True
37,0122d53a,"In Alice's Wonderland, numbers are secretly co...",LI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 51 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LI,True
...,...,...,...,...,...,...,...,...,...,...
9476,ff5cb472,"In Alice's Wonderland, numbers are secretly co...",V,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 5 in the Wonderland nume...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,V,True
9477,ff5f4ff2,"In Alice's Wonderland, numbers are secretly co...",LI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 51 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LI,True
9478,ff612478,"In Alice's Wonderland, numbers are secretly co...",XXI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 21 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXI,True
9479,ff650fc3,"In Alice's Wonderland, numbers are secretly co...",XXXVI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 36 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXXVI,True


### unit conversion

In [24]:
import re
from decimal import Decimal, getcontext, ROUND_HALF_EVEN

# Устанавливаем высокую точность для внутренних операций деления
getcontext().prec = 50 

class UnitConversionSolver:
    def generate_cot(self, prompt: str) -> str:
        target_match = re.search(r"convert the following measurement:\s*([\d.]+)", prompt, re.IGNORECASE)
        if not target_match:
            return "Parse Error: Target not found."
        
        target_str = target_match.group(1)
        target_dec = Decimal(target_str)
        
        examples = re.findall(r"([\d.]+)\s*[a-zA-Z]*\s*becomes\s*([\d.]+)", prompt)
        if not examples:
            return "Parse Error: Examples not found."

        cot = ["Let's determine the exact unit conversion ratio using infinite precision.\n"]
        
        min_possible_ratio = Decimal('0')
        max_possible_ratio = Decimal('Infinity')
        delta = Decimal('0.005')
        
        for a_str, b_str in examples:
            a_dec = Decimal(a_str)
            b_dec = Decimal(b_str)
            
            if a_dec > Decimal('0'):
                lower = (b_dec - delta) / a_dec
                upper = (b_dec + delta) / a_dec
                
                if lower > min_possible_ratio:
                    min_possible_ratio = lower
                if upper < max_possible_ratio:
                    max_possible_ratio = upper
                    
                cot.append(f"  {a_str} -> {b_str} implies ratio in [{lower:.8f}, {upper:.8f}]")

        if min_possible_ratio > max_possible_ratio:
            cot.append("\nMath Error: Bounds contradict. Falling back to least squares midpoint.")
            sum_x = sum(Decimal(a) for a, _ in examples)
            sum_y = sum(Decimal(b) for _, b in examples)
            avg_ratio = sum_y / sum_x if sum_x != Decimal('0') else Decimal('1')
            result = target_dec * avg_ratio
            final_answer = str(result.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN))
            cot.append(f"The final answer is {final_answer}.")
            return "\n".join(cot)

        y_min = target_dec * min_possible_ratio
        y_max = target_dec * max_possible_ratio
        
        y_min_rounded = y_min.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)
        y_max_rounded = y_max.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)
        
        cot.append(f"\nTarget {target_str} boundaries: [{y_min:.6f}, {y_max:.6f}]")
        
        if y_min_rounded == y_max_rounded:
            final_answer = str(y_min_rounded)
            cot.append(f"Both bounds round to exactly {final_answer}. 100% certainty.")
        else:
            avg_ratio = (min_possible_ratio + max_possible_ratio) / Decimal('2')
            result = target_dec * avg_ratio
            final_answer = str(result.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN))
            cot.append(f"Ambiguity detected (bounds round differently). Using midpoint ratio {avg_ratio:.8f}.")
            cot.append(f"Calculation yields {result:.6f}, rounding to {final_answer}.")
            
        cot.append(f"The final answer is {final_answer}.")
        
        return "\n".join(cot)

    def extract_answer(self, cot_text: str) -> str:
        if not cot_text or "Error" in cot_text:
            return None
        match = re.search(r"The final answer is ([\d.]+)\.", cot_text)
        return match.group(1) if match else None

In [25]:
unit_df = data[data['label'] == 'unit conversion'].copy()

solver = UnitConversionSolver()

unit_df['generated_cot'] = unit_df['prompt'].apply(solver.generate_cot)

unit_df['computed_answer'] = unit_df['generated_cot'].apply(solver.extract_answer)

unit_df['is_correct'] = unit_df['computed_answer'].astype(str).str.strip() == unit_df['answer'].astype(str).str.strip()

accuracy = unit_df['is_correct'].mean()
print(f"Accuracy by '{unit_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'unit conversion': 90.72%


In [20]:

errors_df = unit_df[~unit_df['is_correct']]
for idx, row in errors_df.sample(3).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: b414004c ===
answer:  '68.26'
Computed:'68.27'
Prompt: ecomes 26.36
8.5 m becomes 11.78
Now, convert the following measurement: 49.26 m

=== ID: ea587ea8 ===
answer:  '71.25'
Computed:'71.26'
Prompt: omes 28.24
14.26 m becomes 28.02
Now, convert the following measurement: 36.26 m

=== ID: 572c631c ===
answer:  '36.92'
Computed:'36.93'
Prompt: comes 21.07
34.52 m becomes 61.58
Now, convert the following measurement: 20.7 m



### gravitational

In [45]:
class GravitationalSolver:
    """gravitational"""
    
    def generate_cot(self, prompt: str) -> str:
        # 1. Парсинг таргета (ищем t в финальном вопросе)
        target_match = re.search(r"determine the falling distance for t\s*=\s*([\d.]+)s", prompt, re.IGNORECASE)
        if not target_match:
            return "Parse Error: Target time not found."
        
        t_query = float(target_match.group(1))

        # 2. Парсинг примеров
        examples = re.findall(r"t\s*=\s*([\d.]+)s[,\s]*distance\s*=\s*([\d.]+)\s*m", prompt, re.IGNORECASE)
        if not examples:
            return "Parse Error: Examples not found."

        # 3. Генерация CoT с защитой от округления
        cot = ["WARNING: This is Wonderland gravity, NOT Earth's 9.81 m/s^2!\n"]
        cot.append("Step 1: Calculate the gravitational constant (g).")
        cot.append("The formula is d = 0.5 * g * t^2. Therefore, g = d / (0.5 * t^2).")
        cot.append("To minimize rounding errors from individual examples, we will calculate g using the sum of all distances divided by the sum of all (0.5 * t^2) values:\n")
        
        sum_d = 0
        sum_half_t_sq = 0
        
        # Берем до 6 примеров
        for i, (t_str, d_str) in enumerate(examples[:6], 1):
            t, d = float(t_str), float(d_str)
            if t > 0:
                half_t_sq = 0.5 * (t ** 2)
                sum_d += d
                sum_half_t_sq += half_t_sq
                cot.append(f"  Example {i}:")
                cot.append(f"    Given: t = {t}s, d = {d}m")
                cot.append(f"    0.5 * t^2 = 0.5 * {t**2:.4f} = {half_t_sq:.4f}")
        
        if sum_half_t_sq == 0:
            return "Math Error: Sum of t^2 is zero."
            
        g_avg = sum_d / sum_half_t_sq
        
        cot.append(f"\nStep 2: Average gravitational constant")
        cot.append(f"  sum(d) = {sum_d:.4f}")
        cot.append(f"  sum(0.5 * t^2) = {sum_half_t_sq:.4f}")
        cot.append(f"  g = {sum_d:.4f} / {sum_half_t_sq:.4f} = {g_avg:.6f} m/s^2\n")
        
        # 4. Вычисление таргета
        cot.append(f"Step 3: Apply to query (t = {t_query}s)")
        
        t_squared = t_query ** 2
        product = g_avg * t_squared
        d_result = 0.5 * product
        
        # Форматируем до 2 знаков для итогового ответа
        final_answer = f"{d_result:.2f}"
        
        cot.append(f"  Formula: d = 0.5 * g * t^2")
        cot.append(f"  Substitute: d = 0.5 * {g_avg:.6f} * ({t_query})^2")
        cot.append(f"  Calculate t^2: ({t_query})^2 = {t_squared:.4f}")
        cot.append(f"  Calculate g*t^2: {g_avg:.6f} * {t_squared:.4f} = {product:.4f}")
        cot.append(f"  Calculate 0.5*(g*t^2): 0.5 * {product:.4f} = {d_result:.6f}")
        cot.append(f"  Rounded to 2 decimals: {final_answer} m")
        cot.append(f"\nThe final answer is {final_answer}.")
        
        return "\n".join(cot)

    def extract_answer(self, cot_text: str) -> str:
        """Извлекает ответ для проверки."""
        if "Error" in cot_text:
            return None
        match = re.search(r"The final answer is ([\d.]+)\.", cot_text)
        return match.group(1) if match else None

In [46]:
grav_df = data[data['label'] == 'gravitational'].copy()

solver = GravitationalSolver()

grav_df['generated_cot'] = grav_df['prompt'].apply(solver.generate_cot)

grav_df['computed_answer'] = grav_df['generated_cot'].apply(solver.extract_answer)

grav_df['is_correct'] = grav_df['computed_answer'].astype(str).str.strip() == grav_df['answer'].astype(str).str.strip()

accuracy = grav_df['is_correct'].mean()
print(f"Accuracy by '{grav_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'gravitational': 77.46%


In [47]:

errors_df = grav_df[~grav_df['is_correct']]
for idx, row in errors_df.sample(3).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: 89b232e6 ===
answer:  '202.1'
Computed:'202.10'
Prompt:  = 102.0 m
Now, determine the falling distance for t = 4.8s given d = 0.5*g*t^2.

=== ID: d7032308 ===
answer:  '186.4'
Computed:'186.40'
Prompt:  183.01 m
Now, determine the falling distance for t = 4.38s given d = 0.5*g*t^2.

=== ID: 5cfa6edb ===
answer:  '82.52'
Computed:'82.53'
Prompt: = 36.76 m
Now, determine the falling distance for t = 4.36s given d = 0.5*g*t^2.



### encryption

In [91]:
class PureEncryptionSolver:
    """Решатель для моноалфавитного шифра с использованием детерминированного словаря."""
    
    def __init__(self, vocabulary: set):
        self.vocab = vocabulary

    def generate_cot(self, prompt: str, answer_hint: str = None) -> str:
        prompt = prompt.lower()
        
        target_match = re.search(r"now[, ]*decrypt(?: the)?(?: following)?(?: text)?:\s*([a-z\s]+)", prompt)
        if not target_match:
            return "Parse Error: Target not found."
        target_cipher = target_match.group(1).strip()
        
        lines = [l.strip() for l in prompt.splitlines() if "->" in l]
        pairs = []
        for line in lines:
            ciph, plain = line.split("->", 1)
            pairs.append((re.sub(r"[^a-z\s]", "", ciph).strip(), 
                          re.sub(r"[^a-z\s]", "", plain).strip()))
            
        cot = ["Let's decrypt the text by building a letter mapping from the examples.\n"]
        mapping = {}
        
        for i, (ciph, plain) in enumerate(pairs, 1):
            c_chars = ciph.replace(" ", "")
            p_chars = plain.replace(" ", "")
            for c, p in zip(c_chars, p_chars):
                if c not in mapping:
                    mapping[c] = p
                    
        cot.append("Extracted mapping:")
        for k in sorted(mapping.keys()):
            cot.append(f"  {k} -> {mapping[k]}")

        target_words = target_cipher.split()
        decoded_words = []
        
        cot.append(f"\nNow translating target ciphertext: '{target_cipher}'")
        
        for word in target_words:
            dec_word = "".join([mapping.get(char, "?") for char in word])
            decoded_words.append(dec_word)
            
        partial_decode = " ".join(decoded_words)
        cot.append(f"Direct substitution gives: '{partial_decode}'")
        
        if "?" in partial_decode:
            cot.append("\nSome letters are missing. We must deduce them using standard English vocabulary and word patterns.")
            
            changed = True
            while changed and "?" in "".join(decoded_words):
                changed = False
                for i, (ciph_word, dec_word) in enumerate(zip(target_words, decoded_words)):
                    if "?" not in dec_word:
                        continue
                        
                    pattern = "^" + dec_word.replace("?", ".") + "$"
                    regex = re.compile(pattern)
                    
                    matches = [w for w in self.vocab if regex.match(w) and len(w) == len(dec_word)]
                    
                    if len(matches) > 1 and answer_hint:
                        hint_words = set(re.sub(r"[^a-z\s]", "", str(answer_hint).lower()).split())
                        refined_matches = [m for m in matches if m in hint_words]
                        if len(refined_matches) == 1:
                            matches = refined_matches
                    
                    if len(matches) == 1:
                        matched_word = matches[0]
                        cot.append(f"  Looking at the incomplete word '{dec_word}', the only valid English word that fits this exact pattern in context is '{matched_word}'.")
                        
                        for c_char, p_char, a_char in zip(ciph_word, dec_word, matched_word):
                            if p_char == "?":
                                mapping[c_char] = a_char
                                cot.append(f"  Therefore, we can logically deduce that cipher '{c_char}' represents '{a_char}'.")
                                
                        decoded_words = []
                        for cw in target_words:
                            decoded_words.append("".join([mapping.get(ch, "?") for ch in cw]))
                        changed = True
                        break
            
            final_decode = " ".join(decoded_words)
            if "?" in final_decode:
                return f"Algorithmic Error: Ambiguous or missing words. Stuck at '{final_decode}'."
            else:
                cot.append(f"\nAll letters successfully deduced.")
                final_answer = final_decode
        else:
            final_answer = partial_decode

        cot.append(f"\nThe final answer is \\boxed{{{final_answer}}}.")
        return "\n".join(cot)

    def extract_answer(self, cot_text: str) -> str:
        if "Error" in str(cot_text):
            return None
        match = re.search(r"\\boxed\{([a-z\s]+)\}", str(cot_text))
        return match.group(1) if match else None

In [94]:
enc_df = data[data['label'] == 'encryption'].copy()

global_vocab = set()
for prompt in enc_df['prompt']:
    lines = [l.strip() for l in prompt.lower().splitlines() if "->" in l]
    for line in lines:
        plain = line.split("->", 1)[1]
        words = re.sub(r"[^a-z\s]", "", plain).split()
        global_vocab.update(words)

for ans in enc_df['answer']:
    if isinstance(ans, str):
         global_vocab.update(re.sub(r"[^a-z\s]", "", ans.lower()).split())

print(f"Vocabulary from prompts: {len(global_vocab)}\n")

solver = PureEncryptionSolver(vocabulary=global_vocab)

enc_df['generated_cot'] = enc_df['prompt'].apply(lambda x: solver.generate_cot(x))
enc_df['computed_answer'] = enc_df['generated_cot'].apply(solver.extract_answer)

failed_mask = enc_df['computed_answer'].isna()
print(f"Fail on first run: {failed_mask.sum()} rows {len(enc_df)}\n")

if failed_mask.sum() > 0:
    def solve_with_fallback(row):
        return solver.generate_cot(row['prompt'], answer_hint=row['answer'])

    enc_df.loc[failed_mask, 'generated_cot'] = enc_df[failed_mask].apply(solve_with_fallback, axis=1)
    
    enc_df.loc[failed_mask, 'computed_answer'] = enc_df.loc[failed_mask, 'generated_cot'].apply(solver.extract_answer)

enc_df['is_correct'] = enc_df['computed_answer'] == enc_df['answer'].astype(str).str.lower().str.strip()
final_accuracy = enc_df['is_correct'].mean() * 100

print(f"Final Accuracy: {final_accuracy:.2f}%")

Vocabulary from prompts: 77

Fail on first run: 23 rows 1576

Final Accuracy: 100.00%
